In [4]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf 

import numpy as np
from sklearn.preprocessing import normalize

# Prepare the data

In [5]:
game_folder = "data/game_data_with_tags.csv"
time_folder = "data/time_data.csv"

game_data = pd.read_csv(game_folder)
time_data = pd.read_csv(time_folder)

In [6]:
game_time_sum = time_data.groupby('game_id')['user_id'].count()
game_time_sum = game_time_sum.reset_index()
game_time_sum.columns = ['game_id', 'users']
game_time_sum = game_time_sum.sort_values(by='users', ascending=False)
# get ids of games with at least 50 users, and filter the time_data
game_time_sum = game_time_sum[game_time_sum['users'] >= 50]

# get the game ids
game_ids = game_time_sum['game_id']

# filter the game_data
game_data = game_data[game_data['game_id'].isin(game_ids)]
len(game_data)

7606

In [7]:
uniq_tags = []
for row in game_data.itertuples():
    if row.tag_0: uniq_tags.append(row.tag_0)
    if row.tag_1: uniq_tags.append(row.tag_1)
    if row.tag_2: uniq_tags.append(row.tag_2)
    if row.tag_3: uniq_tags.append(row.tag_3)
    if row.tag_4: uniq_tags.append(row.tag_4)
uniq_tags = list(set(uniq_tags))
# remove nan
uniq_tags = [tag for tag in uniq_tags if tag == tag]

# create encoding of tags to integers
tags2int = {k: v for v, k in enumerate(uniq_tags)}

len(uniq_tags)

412

In [8]:
# find rows where tag_0 is nan, remove them (that means the game has no tags)
game_data = game_data[game_data["tag_0"] == game_data["tag_0"]]

# remove all time_data rows where the game_id is not in game_data
time_data = time_data[time_data['game_id'].isin(game_data['game_id'])]

In [9]:
game_tag_vectors = {}
for row in game_data.itertuples():
    game_tag_vectors[row.game_id] = [0] * len(uniq_tags)
    for i, tag in enumerate(uniq_tags):
        if row.tag_0 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_1 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_2 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_3 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_4 == tag: game_tag_vectors[row.game_id][i] = 1

for key in game_tag_vectors:
    game_tag_vectors[key] = normalize([game_tag_vectors[key]], axis=1, norm='l2')[0]

In [10]:
time_data = time_data.sample(frac=1, random_state=42)
train_data = time_data[:int(0.9 * len(time_data))]
val_data = time_data[int(0.9 * len(time_data)):]

In [11]:
uniq_users = list(set(train_data['user_id']))

In [12]:
# make a vector for each game
game_tag_vectors = {}
for row in game_data.itertuples():
    game_tag_vectors[row.game_id] = [0] * len(uniq_tags)
    for i, tag in enumerate(uniq_tags):
        if row.tag_0 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_1 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_2 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_3 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_4 == tag: game_tag_vectors[row.game_id][i] = 1

# normalize the vectors, keep the dictionary
for key in game_tag_vectors:
    game_tag_vectors[key] = normalize([game_tag_vectors[key]], axis=1, norm='l2')[0]

time_data = time_data.sample(frac=1, random_state=42)
train_data = time_data[:int(0.9 * len(time_data))]
val_data = time_data[int(0.9 * len(time_data)):]

uniq_users = list(set(train_data['user_id']))

# Evaluate the model

In [13]:
rec_tally = 0
for iter, user in tqdm(enumerate(uniq_users), total=len(uniq_users)):
    user_data = train_data[train_data['user_id'] == user]

    played_tags = {}

    for row in user_data.itertuples():
        game_tags = game_data.loc[game_data["game_id"] == row.game_id, "tag_0":"tag_4"].values[0]
        for tag in game_tags:
            # check if tag is nan
            if tag != tag:
                continue
            played_tags[tag] = played_tags.get(tag, 0) + row.playtime

    user_tags_vector = np.zeros(len(uniq_tags))
    user_id = 1
    for item in played_tags:
        user_tags_vector[tags2int[item]] = played_tags[item]

    user_tags_vector = normalize([user_tags_vector], norm='l2')[0]

    # compute pairwise cosine similarity
    cosine_similarities = {}

    for key in game_tag_vectors:
        game_vector = game_tag_vectors[key]
        cosine_similarities[key] = np.dot(user_tags_vector, game_vector)

    # get rid of games that the user has already played
    for game_id in user_data["game_id"]:
        cosine_similarities.pop(game_id, None)

    recommendations = sorted(cosine_similarities.items(),
                            key=lambda x: x[1], reverse=True)[:20]
    recommendations = [x[0] for x in recommendations]


    # check how many of the recommendations are in the validation set
    val_games = set(val_data[val_data['user_id'] == user]['game_id'])
    rec_tally += len(set(recommendations).intersection(val_games))

print("Total recommendations in validation set:", rec_tally)
print("Total users:", len(uniq_users))
print("Recommendations per user:", rec_tally / len(uniq_users))

  0%|          | 0/6571 [00:00<?, ?it/s]

Total recommendations in validation set: 1915
Total users: 6571
Recommendations per user: 0.2914320499162989


# Generate game recommendations for random user

In [20]:
# take a random user
user_id = np.random.choice(uniq_users)
user_time_data = time_data[time_data['user_id'] == user_id]
user_time_data

,user_id,game_id,playtime
980469,2030,937,1
980486,2030,4359,319
980478,2030,8,181
980480,2030,7152,7042
980473,2030,13,61
980483,2030,2729,126
980471,2030,1,236
980477,2030,1398,28
980481,2030,2760,15913
980475,2030,9108,4732


In [21]:
played_tags = {}

for row in user_time_data.itertuples():
    game_tags = game_data.loc[game_data["game_id"] == row.game_id, "tag_0":"tag_4"].values[0]
    for tag in game_tags:
        # check if tag is nan
        if tag != tag:
            continue
        played_tags[tag] = played_tags.get(tag, 0) + row.playtime

In [22]:
user_tags_vector = np.zeros(len(uniq_tags))
user_id = 1
for item in played_tags:
    user_tags_vector[tags2int[item]] = played_tags[item]

user_tags_vector = normalize([user_tags_vector], norm='l2')
user_tags_tensor = tf.convert_to_tensor(user_tags_vector, dtype=tf.float32)

In [23]:
# compute pairwise cosine similarity
cosine_similarities = {}

for key in game_tag_vectors:
    game_vector = game_tag_vectors[key]
    game_vector_tensor = tf.convert_to_tensor([game_vector], dtype=tf.float32)
    cosine_similarities[key] = tf.keras.losses.cosine_similarity(user_tags_tensor, game_vector_tensor).numpy()[0]

# get rid of games that the user has already played
for game_id in user_time_data["game_id"]:
    cosine_similarities.pop(game_id, None)

In [24]:
# extract 20 most similar games, and their ids
recommendations = sorted(cosine_similarities.items(), key=lambda x: x[1], reverse=False)[:20]

In [25]:
# get the game names and tags
game_names = []
tags = []
for game_id, _ in recommendations:
    game_row = game_data.loc[game_data["game_id"] == game_id]
    game_names.append(game_row["game_name"].values[0])
    game_tags = []
    for i in range(5):
        if game_row[f"tag_{i}"].values[0]: game_tags.append(game_row[f"tag_{i}"].values[0])
    tags.append(game_tags)
for item in zip(game_names, tags):
    print(item)

('Deathmatch Classic', ['Action', 'FPS', 'Classic', 'Multiplayer', 'Shooter'])
('Homefront: The Revolution', ['Action', 'FPS', 'Open World', 'Shooter', 'Multiplayer'])
('Call of Duty: Black Ops III', ['Multiplayer', 'Zombies', 'FPS', 'Shooter', 'Action'])
('Call of Duty: Black Ops', ['Action', 'FPS', 'Zombies', 'Multiplayer', 'Shooter'])
('Call of Duty: Black Ops - Multiplayer', ['Action', 'FPS', 'Zombies', 'Multiplayer', 'Shooter'])
('Day of Defeat: Source', ['FPS', 'World War II', 'Multiplayer', 'Action', 'Shooter'])
('Wolfenstein: Enemy Territory', ['FPS', 'World War II', 'Multiplayer', 'Action', 'Shooter'])
('unknown game', ['Free to Play', 'Action', 'Multiplayer', 'FPS', 'Shooter'])
('Blacklight: Retribution', ['Free to Play', 'FPS', 'Multiplayer', 'Action', 'Shooter'])
('unknown game', ['Free to Play', 'Shooter', 'FPS', 'Multiplayer', 'Action'])
('Tribes: Ascend', ['Free to Play', 'FPS', 'Action', 'Multiplayer', 'Shooter'])
('Block N Load', ['Free to Play', 'Action', 'Multiplayer